# RetailMind AI ? Day 7: Transfer Learning

This notebook contains the complete ResNet50 training code. It predicts the catalog?s `masterCategory` labels, first training a new output layer and then fine-tuning ResNet50. YOLO/object detection is not included.

## Before starting

Put `styles.csv` and `images/` in `data/fashion-product-images-small/`. ResNet50 downloads pretrained weights on first use, so internet access is required. Training on all photos can take a long time.

In [ ]:
from pathlib import Path
import csv, json, random
import numpy as np
import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / "data" / "fashion-product-images-small"
MODEL_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports"
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
IMAGE_SIZE = (128, 128)
BATCH_SIZE = 32
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Read image paths and the masterCategory label.
csv_path = DATA_DIR / "styles.csv"
if not csv_path.is_file(): csv_path = DATA_DIR / "myntradataset" / "styles.csv"
image_dir = csv_path.parent / "images"
if not csv_path.is_file() or not image_dir.is_dir(): raise FileNotFoundError("Put styles.csv and images/ in data/fashion-product-images-small/.")
# Read rows with Python's CSV reader so quoted commas and extra fields are handled.
records = []
with csv_path.open("r", encoding="utf-8", newline="") as file:
    for row in csv.DictReader(file):
        image_id = (row.get("id") or "").strip()
        category = (row.get("masterCategory") or "").strip()
        image_path = image_dir / (image_id + ".jpg")
        if image_id and category and image_path.is_file():
            records.append({"image_path": str(image_path), "masterCategory": category})
styles = pd.DataFrame(records)
if styles.empty: raise ValueError("No valid catalog image rows were found.")
counts = styles["masterCategory"].value_counts()
styles = styles[styles["masterCategory"].isin(counts[counts >= 3].index)].copy()
class_names = sorted(styles["masterCategory"].unique())
label_id = {name: index for index, name in enumerate(class_names)}
styles["label"] = styles["masterCategory"].map(label_id)

# Split each category into training, validation, and test photos.
rng = random.Random(SEED)
train_rows, valid_rows, test_rows = [], [], []
for _, group in styles.groupby("masterCategory"):
    rows = group.to_dict("records")
    rng.shuffle(rows)
    n_test = max(1, round(len(rows) * 0.15))
    n_valid = max(1, round(len(rows) * 0.15))
    while n_test + n_valid >= len(rows):
        if n_test >= n_valid and n_test > 1: n_test -= 1
        elif n_valid > 1: n_valid -= 1
        else: break
    test_rows.extend(rows[:n_test])
    valid_rows.extend(rows[n_test:n_test+n_valid])
    train_rows.extend(rows[n_test+n_valid:])
print("Categories:", class_names)
print("Train:", len(train_rows), "Validation:", len(valid_rows), "Test:", len(test_rows))

def make_dataset(rows, training=False):
    paths = [row["image_path"] for row in rows]
    labels = [row["label"] for row in rows]
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training: dataset = dataset.shuffle(min(len(rows), 5000), seed=SEED)
    def load_image(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        image = tf.image.resize(image, IMAGE_SIZE)
        if training: image = tf.image.random_flip_left_right(image)
        return tf.keras.applications.resnet50.preprocess_input(image), label
    return dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
train_ds = make_dataset(train_rows, training=True)
valid_ds = make_dataset(valid_rows)
test_ds = make_dataset(test_rows)

## Train and evaluate

The model is tested on product photos kept out of training. The code below saves the model, test accuracy, and confusion matrix.

In [ ]:
# Load ResNet50 image features and freeze them at first.
backbone = tf.keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=(128, 128, 3))
backbone.trainable = False
inputs = tf.keras.Input(shape=(128, 128, 3))
features = backbone(inputs, training=False)
features = tf.keras.layers.GlobalAveragePooling2D()(features)
features = tf.keras.layers.Dropout(0.3)(features)
outputs = tf.keras.layers.Dense(len(class_names), activation="softmax")(features)
model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])

# Stage 1: train the new category classifier.
model.fit(train_ds, validation_data=valid_ds, epochs=5, callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)], verbose=2)

# Stage 2: fine-tune the last ResNet block using a smaller learning rate.
backbone.trainable = True
for layer in backbone.layers:
    layer.trainable = layer.name.startswith("conv5_") and not isinstance(layer, tf.keras.layers.BatchNormalization)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, validation_data=valid_ds, epochs=3, callbacks=[tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)], verbose=2)

# Test on photos that were not used for training.
test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
print("Day 7 test accuracy:", round(float(test_accuracy), 4))
model.save(MODEL_DIR / "vision_master_category_resnet50_notebook.keras")
(MODEL_DIR / "vision_master_category_resnet50_notebook_classes.json").write_text(json.dumps(class_names, indent=2), encoding="utf-8")
pd.DataFrame([{"test_loss": test_loss, "test_accuracy": test_accuracy}]).to_csv(REPORT_DIR / "day7_transfer_notebook_metrics.csv", index=False)

# Save a confusion matrix to show which categories were mixed up.
true_labels = np.concatenate([labels.numpy() for _, labels in test_ds])
predicted_labels = model.predict(test_ds, verbose=0).argmax(axis=1)
confusion = tf.math.confusion_matrix(true_labels, predicted_labels, num_classes=len(class_names)).numpy()
pd.DataFrame(confusion, index=class_names, columns=class_names).to_csv(REPORT_DIR / "day7_master_category_notebook_confusion.csv")

In [ ]:
# If Day 6 has been trained, compare it with ResNet50 on the same test photos.
day6_path = MODEL_DIR / "vision_master_category_notebook.keras"
day6_names_path = MODEL_DIR / "vision_master_category_notebook_classes.json"
if not day6_path.is_file():
    day6_path = MODEL_DIR / "vision_master_category_classifier.keras"
    day6_names_path = MODEL_DIR / "vision_master_category_class_names.json"
if day6_path.is_file() and day6_names_path.is_file():
    saved_names = json.loads(day6_names_path.read_text(encoding="utf-8"))
    if saved_names != class_names:
        raise ValueError("Day 6 and Day 7 category lists differ. Retrain Day 6 first.")
    raw_test = tf.data.Dataset.from_tensor_slices((
        [row["image_path"] for row in test_rows],
        [row["label"] for row in test_rows],
    ))
    def load_raw_image(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        return tf.image.resize(image, IMAGE_SIZE), label
    raw_test = raw_test.map(load_raw_image).batch(BATCH_SIZE)
    day6_model = tf.keras.models.load_model(day6_path)
    day6_loss, day6_accuracy = day6_model.evaluate(raw_test, verbose=0)
    comparison = pd.DataFrame([
        {"Model": "Day 6 catalog CNN", "Test accuracy": day6_accuracy},
        {"Model": "Day 7 ResNet50", "Test accuracy": test_accuracy},
    ])
    display(comparison)
    comparison.to_csv(REPORT_DIR / "day7_notebook_comparison.csv", index=False)
else:
    print("To compare scores, train the Day 6 catalog CNN first.")

## Saved files

- Model and labels: `models/vision_master_category_resnet50_notebook.keras`, `models/vision_master_category_resnet50_notebook_classes.json`
- Test accuracy and loss: `reports/day7_transfer_notebook_metrics.csv`
- Confusion matrix: `reports/day7_master_category_notebook_confusion.csv`